In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime as dt
from utils.physics.sound_model.ellipsoidal_sound_model import GridEllipsoidalSoundModel

In [ ]:
from pathlib import Path


# --- Fonctions de chargement des dorsales ---
def load_ridge_data(dorsal_db_path: str):
    """
    Charge les données des dorsales océaniques à partir de fichiers .xy.
    """
    dorsal_db_path = Path(dorsal_db_path)
    if not dorsal_db_path.is_dir():
        raise FileNotFoundError(f"Le dossier {dorsal_db_path} n'existe pas.")

    dorsal_files = [f for f in dorsal_db_path.glob('*.xy')]
    if not dorsal_files:
        raise ValueError(f"Aucun fichier .xy trouvé dans {dorsal_db_path}.")

    print(f"Chargement de {len(dorsal_files)} fichiers de dorsales: {[f.name for f in dorsal_files]}")

    ridge_data = {}
    all_ridge_points = []

    for f in dorsal_files:
        try:
            ridge_name = f.stem.replace('axe-', '')
            df = pd.read_csv(f, comment=">", sep=r'\s+')
            ridge_points = df[['y', 'x']].values  # Assurez-vous que 'y' est la latitude et 'x' la longitude

            ridge_data[ridge_name] = ridge_points
            all_ridge_points.append(ridge_points)

            print(f"  {ridge_name}: {len(ridge_points)} points")
        except Exception as e:
            print(f"Erreur lors du chargement de {f.name}: {e}")
            continue

    if not ridge_data:
        raise ValueError("Aucune donnée valide n'a pu être chargée.")

    all_ridge_points = np.vstack(all_ridge_points)
    print(f"Nombre total de points de dorsales: {len(all_ridge_points)}")

    return ridge_data, all_ridge_points

ridge_data_path = "../../data/dorsales/"
_, ridge_points = load_ridge_data(ridge_data_path)


In [ ]:
PATH = '/media/rsafran/CORSAIR/Catalogue_vaibhav/01_SWIR_swarms/02_2018_Jul_31S_Novara_TF/'
cat_file_name_isc = 'H_2018_Jul_31S_ISC'
cat_file_name_imp= 'H_2018_Jul_31S_IMP'
cat_file_name_cmt= 'H_2018_Jul_31S_CMT'
cat_file_name_oha= 'H_2018_Jul_31S_OHA'
sta_file = "2018_Network_Geometry.csv"

In [ ]:
names = ["datetime", 'nb_stations', 'stations_ID', 'lat', 'lon', 'lat_err_km','lon_err_km','time_err', 'RL','RL_sup','code']
#=========================================
#code explaination :
# eq -T-wave signal is strong and clearly visible on at least 4 hydrophones;
# eqm - T-wave signal is very weak;
# eqisc - assigned to hydroacoustically relocated ISC catalog event;
# eqcmt - same as previous but for which GCMT solution is available;
# eqi - “impulsive events” as lava-water interactions+

data_isc = pd.read_csv(PATH+cat_file_name_isc,sep =r'\s+' ,names=names,parse_dates=['datetime'],date_format='ISO8601')
data_imp = pd.read_csv(PATH+cat_file_name_imp,sep =r'\s+' ,names=names,parse_dates=['datetime'],date_format='ISO8601')
data_cmt = pd.read_csv(PATH+cat_file_name_cmt,sep =r'\s+' ,names=names,parse_dates=['datetime'],date_format='ISO8601')
data_oha = pd.read_csv(PATH+cat_file_name_oha,sep =r'\s+' ,names=names,parse_dates=['datetime'],date_format='ISO8601')

In [ ]:
data_all = pd.concat([data_cmt,data_oha,data_imp,data_isc]).reset_index(drop=True)
data_all["datetime"] =  pd.to_datetime(data_all["datetime"],utc=True)
data_all

In [ ]:
stations = pd.read_csv(PATH+sta_file, sep='\t',comment='#',index_col=0, header=None)


In [ ]:
my_loc_path ='/home/rsafran/PycharmProjects/toolbox/data/SWIR_CRISE_localized.csv'
my_loc_df = pd.read_csv(my_loc_path,parse_dates=['origin_time'],date_format='ISO8601')
my_loc_df
# my_loc_df["origin_time"] =  pd.to_datetime(my_loc_df["origin_time"],utc=True)

In [ ]:
plt.figure()


ax = plt.gca()
# data_isc.plot(x='lon',y='lat',kind='scatter',s=10,figsize=(10,10),title='SWIR',ax=ax,color='lightgreen',alpha=0.5,label='vaibhav isc')
# data_oha.plot(x='lon',y='lat',kind='scatter',s=10,figsize=(10,10),title='SWIR',ax=ax,color='blue',alpha=0.2,label='vaibhav oha')
# data_cmt.plot(x='lon',y='lat',kind='scatter',s=10,figsize=(10,10),title='SWIR',ax=ax,color='k',alpha=0.5,label='vaibhav cmt')
# data_imp.plot(x='lon',y='lat',kind='scatter',s=50,marker="+",figsize=(10,10),title='SWIR',ax=ax,color='orange',alpha=0.8,label='vaibhav imp')
data_all.plot(x='lon',y='lat',kind='scatter',s=10,marker="o",figsize=(10,10),title='SWIR',ax=ax,color='darkblue',alpha=0.2,label='vaibhav all')
my_loc_df.plot(x='lon',y='lat',kind='scatter',s=10,figsize=(10,10),title='SWIR',ax=ax, color='red',alpha=0.2,label = 'romain')

# Tracer la dorsale
ridge_lats = ridge_points[:, 0]  # Latitude
ridge_lons = ridge_points[:, 1]  # Longitude
plt.plot(ridge_lons, ridge_lats, '--', linewidth=1,color='k', label='Dorsale',zorder=0 )

(lat_min, lat_max),(lon_min, lon_max) =(np.float64(-32.22384054054054), np.float64(-31.123840540540606)) ,(np.float64(57.75945945945946), np.float64(58.859459459459394))
plt.xlim(lon_min, lon_max)
plt.ylim(lat_min,lat_max)

In [ ]:
#look at matches

data_all.sort_values('datetime',inplace=True)
my_loc_df.sort_values('origin_time', inplace=True)
tmp = pd.merge_asof(
    data_all,
    my_loc_df,
    left_on='datetime',
    right_on='origin_time',
    tolerance=pd.Timedelta(seconds=10),
    direction='nearest',

)
tmp.dropna(inplace=True)
print(len(tmp))

In [ ]:
tmp.dropna(inplace=True)
print(len(tmp))

In [ ]:
plt.figure()


ax = plt.gca()
# data_isc.plot(x='lon',y='lat',kind='scatter',s=10,figsize=(10,10),ax=ax,color='lightgreen',alpha=0.5,label='vaibhav isc')
# data_oha.plot(x='lon',y='lat',kind='scatter',s=10,figsize=(10,10),ax=ax,color='blue',alpha=0.2,label='vaibhav oha')
# data_cmt.plot(x='lon',y='lat',kind='scatter',s=10,figsize=(10,10),ax=ax,color='k',alpha=0.5,label='vaibhav cmt')
# data_imp.plot(x='lon',y='lat',kind='scatter',s=50,marker="+",figsize=(10,10),ax=ax,color='orange',alpha=0.8,label='vaibhav imp')
# data_all.plot(x='lon',y='lat',kind='scatter',s=10,marker="o",figsize=(10,10),ax=ax,color='darkblue',alpha=0.2,label='vaibhav all')
# my_loc_df.plot(x='lon',y='lat',kind='scatter',s=10,figsize=(10,10),ax=ax, color='gray',alpha=0.2,label = 'romain')
tmp.plot(x='lon_x',y='lat_x',kind='scatter',s=10,figsize=(10,10), ax=ax ,color='red',alpha=0.5,label = 'matches loc vaibhav')
tmp.plot(x='lon_y',y='lat_y',kind='scatter',s=10,figsize=(10,10), ax=ax ,color='orange',alpha=0.5,label = 'matches loc romain')
# Tracer la dorsale
ridge_lats = ridge_points[:, 0]  # Latitude
ridge_lons = ridge_points[:, 1]  # Longitude
plt.plot(ridge_lons, ridge_lats, '--', linewidth=1,color='k', label='Dorsale',zorder=0 )

(lat_min, lat_max),(lon_min, lon_max) =(np.float64(-32.22384054054054), np.float64(-31.123840540540606)) ,(np.float64(57.75945945945946), np.float64(58.859459459459394))
plt.xlim(lon_min, lon_max)
plt.ylim(lat_min,lat_max)




In [ ]:
tmp['dist_xy'] =np.sqrt((tmp['lat_x']-tmp['lat_y'])**2+(tmp['lon_x']-tmp['lon_y'])**2)

In [ ]:
plt.hist(tmp['dist_xy'],bins=50,color='blue',edgecolor='black',label='hist')

# Load Associations

In [ ]:
text_path = 'H_2018_Jul_31S_catalog.pick'
with open(PATH+text_path,'r') as f:
    for line in f:
        pass
        # print(line.split())
#[nbsta]
#[sta_lat]*nbsta
#[sta_lon]*nbsta
#[sound_velocity_sta]*nbsta
# [unix time, year, julian date (dhms.sss), RL] => this nbsta lines
# [weight from least square] * min(nsta,5)
# [weight from least square] * nsta%5
# [event unix time, event lat, event lon]
#["datetime yyyydddhhmmss.s", 'nb_stations', 'stations_ID', 'lat', 'lon', 'lat_err_km','lon_err_km','time_err', 'RL','RL_sup','code']


In [ ]:
# Exemple d'utilisation

def decode_julian_datetime(julian_datetime_str):
    # Décomposer la chaîne de caractères en année, jour julien, heure, minute, seconde, milliseconde
    if '.' in julian_datetime_str:
        date = dt.datetime.strptime(julian_datetime_str, '%Y%j%H%M%S.%f')
        return date
    else :
        date = dt.datetime.strptime(julian_datetime_str, '%Y%j%H%M%S%f')
    return date


file_path = PATH +'H_2018_Jul_31S_catalog.pick'
# events = extract_event_data(file_path)

with open(file_path, 'r') as f:
    lines = f.readlines()

events = []
i = 0
while i < len(lines):
    line = lines[i].strip()
    if line.isdigit():  # Début d'un nouvel événement
        nbsta = int(line)
        i += 1

        # Sauter les lignes de coordonnées et de vitesse du son
        i += 3

        # Lire les unix times at station
        datetime_sta = []

        for _ in range(nbsta):
            time_line = lines[i].strip().split()
            if len(time_line) ==5:
                time_str = time_line[1]+time_line[2]+'0'+time_line[3]
            else :
                time_str = time_line[1]+time_line[2]
            time=decode_julian_datetime(time_str)
            datetime_sta.append(time)
            i += 1

        # Sauter les lignes de poids (weights)
        i += 2 if nbsta > 5 else 1

        # Lire la ligne de localisation de l'événement
        event_line = lines[i].strip().split()
        event_mystery_time = float(event_line[0])
        event_lat = float(event_line[1])
        event_lon = float(event_line[2])
        i += 1

        # Lire la ligne de métadonnées pour extraire station_ID et datetime
        metadata_line = lines[i].strip().split()
        datetime_str = metadata_line[0].strip()
        station_id = metadata_line[2].strip()
        event_datetime = decode_julian_datetime(datetime_str)
        i += 1

        # Ajouter les données extraites à la liste des événements
        events.append({
            'station_ID': station_id,
            'detection_times_at_station': datetime_sta,
            'event_mystery1_time': event_mystery_time,
            'event_datetime': event_datetime,
            'event_lat': event_lat,
            'event_lon': event_lon,
        })

catalogue_vaihbav = pd.DataFrame(events)

# Affichage du DataFrame
catalogue_vaihbav['station_ID'] = catalogue_vaihbav.station_ID.apply(list)
# catalogue_vaihbav = catalogue_vaihbav.explode(['station_ID', 'detection_times_at_station']).copy()
catalogue_vaihbav



## Reloc Vaibhav

In [ ]:
import os
from utils.data_reading.sound_data.station import StationsCatalog

ISAS_PATH = "/media/rsafran/CORSAIR/ISAS/extracted/2018"
arr = os.listdir(ISAS_PATH)
file_list = [os.path.join(ISAS_PATH, fname) for fname in arr if fname.endswith('.nc')]
SOUND_MODEL = GridEllipsoidalSoundModel(file_list)

CATALOG_PATH = "/media/rsafran/CORSAIR/OHASISBIO/recensement_stations_OHASISBIO_RS.csv"
STATIONS = StationsCatalog(CATALOG_PATH).filter_out_undated().filter_out_unlocated()



In [ ]:
st_obj = {}
for st in STATIONS.by_dataset('OHASISBIO-2018'):
    st_obj[st.name]= st
print(st_obj)

In [ ]:
stations = pd.read_csv(PATH+sta_file, sep='\t',comment='#',index_col=0, header=None).transpose()
stations

In [ ]:
stations[stations['Station_Code']=='E']

In [ ]:
st_trad = {'RTJ': 'RTJ',
 'NE-AMS': 'NEAMS',
 'S-SEIR': 'SSEIR',
 'SW-AMS': 'SWAMS-bot',
 'ELAN': 'ELAN',
 'WKER': 'WKER2',
 'S-SWIR': 'SSWIR',
 'MAD-W': 'MADW',
 'MAD-E': 'MADE'}


In [ ]:
stations['name'] =  stations.Stations.replace(st_trad)
stations['st_obj'] = stations.name.replace(st_obj)

In [ ]:
st_code_to_obj = stations[['Station_Code','st_obj']].set_index('Station_Code').to_dict(orient='dict')["st_obj"]
st_code_to_obj

In [ ]:
tmp = catalogue_vaihbav.explode(['station_ID'])
tmp['st_obj']=tmp['station_ID'].replace(st_code_to_obj)
# Group back and aggregate as lists
catalogue_vaihbav = tmp.groupby(level=0).agg({
    'station_ID': list,
    'st_obj': list,
    **{col: 'first' for col in tmp.columns if col not in ['station_ID', 'st_obj']}
})
catalogue_vaihbav

In [ ]:
list(catalogue_vaihbav.loc[0,['detection_times_at_station']].values)[0]


In [ ]:
catalogue_vaihbav.st_obj.loc[0][0].other_kwargs

In [ ]:

# drift_ppm = {'ELAN':  np.float64(-0.0222),
#              'H01W1': np.float64(0.0),
#              'H04N1': np.float64(0.0),
#              'H08S1': np.float64(0.0),
#              'MADE':  np.float64(0.0),
#              'MADW':  np.float64(0.0),
#              'NEAMS': np.float64(0.0),
#              'RTJ':   np.float64(0.0),
#              'SSEIR': np.float64(0.0),
#              'SSWIR': np.float64(0.0346),
#              'SWAMS-bot': np.float64(0.0048),
#              'WKER2': np.float64(0.0),
#              'H04S1': np.float64(0.0)}

drift_ppm = {'ELAN': np.float64(-0.131710549912667),
         'H01W1': np.float64(0.08420817976058004),
         'H04N1': np.float64(0.005474507338212975),
         'H04S1': np.float64(0.04367580175236678),
         'H08S1': np.float64(0.0339800345578377),
         'MADE': np.float64(0.06750201481222673),
         'MADW': np.float64(0.13357977320875622),
         'NEAMS': np.float64(0.12430695166652732),
         'RTJ': np.float64(0.060539097101553196),
         'SSEIR': np.float64(-0.036162831789924375),
         'SSWIR': np.float64(0.02191309789424821),
         'SWAMS-bot': np.float64(0.0),
         'WKER2': np.float64(0.07208521358944064)}


offset = {'ELAN': np.float64(-1.2),
          'H01W1': np.float64(0),
          'H04N1': np.float64(6.5),
          'H08S1': np.float64(0.0),
          'MADE': np.float64(0.0),
          'MADW': np.float64(0.0),
          'NEAMS': np.float64(0.0),
          'RTJ': np.float64(0.0),
          'SSEIR': np.float64(0.0),
          'SSWIR': np.float64(0.0),
          'SWAMS-bot': np.float64(0.),
          'WKER2': np.float64(0.)}

In [ ]:
station_obj =list(catalogue_vaihbav.loc[0,['st_obj']].values)[0]
det = list(catalogue_vaihbav.loc[0,['detection_times_at_station']].values)[0]
list(map(lambda j,k :dt.timedelta(seconds=offset[j.name]) + dt.timedelta(seconds=j.get_clock_error(k, drift_ppm=drift_ppm[j.name])),station_obj, det ))
list(map(
    lambda s, d: 0.1 + s.get_clock_error(d,drift_ppm=0.28) if "not_ok"  in s.other_kwargs.values() #or ok
                 else 0.1,
    station_obj,
    det
))

detections_uncertanty = list(map(lambda s :  1 if "not_ok"  in s.other_kwargs.values() else 0.5 if "ok"  in s.other_kwargs.values() else 3, station_obj))
detections_uncertanty

In [ ]:
station = list(map(lambda x : x.get_pos(), catalogue_vaihbav.loc[0,['st_obj']].values[0]))
c0 = list(catalogue_vaihbav.loc[0,['event_lat','event_lon']])
min_date = np.argmin(det)
max_date = np.argmax(det)
t0 = -1 * SOUND_MODEL.get_sound_travel_time(c0, station[min_date], det[min_date])
x0 = [t0]+list(c0)

In [ ]:
import multiprocessing as mp
from tqdm import tqdm


def process(i):
    station_obj =list(catalogue_vaihbav.loc[i,['st_obj']].values)[0]
    station = list(map(lambda x : x.get_pos(), catalogue_vaihbav.loc[i,['st_obj']].values[0]))

    if len(station) < 6:
        return None

    det = list(catalogue_vaihbav.loc[i,['detection_times_at_station']].values)[0]
    # drift_errors = list(map(lambda j,k : timedelta(seconds=STATIONS[j].get_clock_error(k, drift_ppm=drift_mesured_10[STATIONS[j].name]))+timedelta(seconds=6.8) if STATIONS[j].name =="H04N1" else timedelta(seconds=STATIONS[j].get_clock_error(k, drift_ppm=drift_mesured_10[STATIONS[j].name])) ,associations[i][0][:,0], det))
    drift_errors = list(map(lambda j,k : dt.timedelta(seconds=offset[j.name]) + dt.timedelta(seconds=j.get_clock_error(k, drift_ppm=drift_ppm[j.name]))
                            - dt.timedelta(seconds=j.get_clock_error(k)) ,station_obj, det ))     #correction des drifts ; -drift : reverse old drift correction
    det = list(map(lambda d, e : d-e,det,drift_errors))

    # drift = list(map(
    #     lambda s, d: 0.1 + STATIONS[s].get_clock_error(IDX_TO_DET[d][0],drift_ppm=0.28) if "not_ok" in STATIONS[s].other_kwargs.values()
    #                  else 0.1,
    #
    #     associations[i][0][:,0],
    #     associations[i][0][:,1]
    # ))
    drift = list(map(
        lambda s, d: 0.1 + s.get_clock_error(d,drift_ppm=0.28) if "not_ok"  in s.other_kwargs.values() #or ok
                     else 0.1,
        station_obj,
        det
    ))
    # drift = list(map(
    # lambda s, d: 0.25*1e-6*d if "not_ok" in STATIONS[s].other_kwargs.values()
    #              else 0,
    # associations[i][0][:,0],
    # associations[i][0][:,1]
    # ))
    drift =  np.abs(drift) / np.sqrt(3)
    # detections_uncertanty = [2]*len(det)
    # detections_uncertanty = list(map(
    #     lambda s :  3 if "not_ok"  in STATIONS[s].other_kwargs.values()
    #                 else 1 if "ok"  in STATIONS[s].other_kwargs.values()
    #                 else 5, associations[i][0][:,0]
    # ))
    #iter2
    detections_uncertanty = list(map(
        lambda s :  1 if "not_ok"  in s.other_kwargs.values()
                    else 0.5 if "ok"  in s.other_kwargs.values()
                    else 3, station_obj
    ))

    c0 = catalogue_vaihbav.loc[i,['event_lat','event_lon']].values
    min_date = np.argmin(det)
    max_date = np.argmax(det)
    t0 = -1 * SOUND_MODEL.get_sound_travel_time(c0, station[min_date], det[min_date])
    x0 = [t0]+list(c0)
    # dmax= SOUND_MODEL.get_distance(np.mean(c0, axis =0), station[max_date])
    # detections_uncertanty = list(map(lambda x : 0.53+SOUND_MODEL.get_distance(np.mean(c0, axis =0), x)/dmax, station))

    res= SOUND_MODEL.localize_with_uncertainties(
        station, det,y_min=lon_min-6, x_min=lat_min-6,y_max=lon_max+6,x_max=lat_max+6, drift_uncertainties=drift,pick_uncertainties=detections_uncertanty, initial_pos=x0
    )

    return i, res
from itertools import compress

#
# # # # Taille du chunk (param important à ajuster selon ton CPU/RAM)
CHUNK_SIZE = 50
results = {}
with mp.Pool(mp.cpu_count()-5) as pool :
    for r in tqdm(pool.imap(process, range(len(catalogue_vaihbav)), chunksize=CHUNK_SIZE),
                  total=len(catalogue_vaihbav)):
        if r is not None:
            i, res = r
            results[i] = res



In [ ]:
results

In [ ]:
quality = []
rows = []
for i in list(results):
    res = results[i]
    rows.append({
        "lat": res.x[1],
        "lon": res.x[2],
        "origin_time": res.x[0],
        "cost": res.cost
    })
    # quality.append(analyze_residuals(res))+

# quality = pd.DataFrame(quality)
reloc = pd.DataFrame(rows)
reloc["origin_time"] = pd.to_datetime(reloc["origin_time"], unit="s",utc=True)

In [ ]:
plt.figure(figsize=(12,8))

plt.subplot(2,2,1)
plt.title('relocalisation vieux drift annulé + drift iter 2 + offset elan')
plt.scatter(reloc['lon'],reloc['lat'],s=10,c=reloc['cost'],alpha=0.8,label='reloc', vmin=0, vmax=10)
# plt.scatter(catalogue_vaihbav['event_lon'],catalogue_vaihbav['event_lat'],s=1,color= 'darkred',alpha=0.1,label='vaibhav')
plt.colorbar()
plt.subplot(222)
plt.gca().set_axis_off()
plt.text(0.0,0.25,str(reloc['cost'].describe()))
plt.subplot(224)
plt.hist(reloc['cost'], bins=100)

In [ ]:
#look at matches

reloc.sort_values('origin_time',inplace=True)
my_loc_df.sort_values('origin_time', inplace=True)
tmp = pd.merge_asof(
    reloc,
    my_loc_df,
    on='origin_time',
    tolerance=pd.Timedelta(seconds=60),
    direction='nearest',

)
tmp.dropna(inplace=True)
tmp['diff_lat']=tmp['lat_x']-tmp['lat_y']
tmp['diff_lon']=tmp['lon_x']-tmp['lon_y']
tmp = tmp[(tmp['diff_lon']<0.1) & (tmp['diff_lon']<0.1)]
print(len(tmp))

plt.scatter(tmp['lon_x'],tmp['lat_x'],s=10,c='green',alpha=0.8,label='reloc', vmin=0, vmax=10)
# plt.scatter(catalogue_vaihbav['event_lon'],catalogue_vaihbav['event_lat'],s=1,color= 'darkred',alpha=0.1,label='vaibhav')
plt.scatter(my_loc_df['lon'],my_loc_df['lat'],s=1,color= 'darkblue',alpha=0.1,label='my_loc')

In [ ]:
print(len(tmp)/len(my_loc_df)*100)

## Plot Bathy and points

In [ ]:
from scipy.io import netcdf_file
from matplotlib.colors import LightSource
from matplotlib.colors import Normalize
import matplotlib.cm as cm

ls = LightSource(azdeg=315, altdeg=45)
cmap =plt.cm.jet

#laoding grid
nc_path = "/media/rsafran/CORSAIR/Documents/Qgis/SWIR_Bathy_Sauter/96200.grd"
data_bathy = netcdf_file(nc_path,'r',mmap=False)
shape = data_bathy.variables['dimension'][::-1]
extent = np.array([data_bathy.variables['x_range'][:],data_bathy.variables['y_range'][:]]).flatten()

x = np.arange(data_bathy.variables['x_range'][0],data_bathy.variables['x_range'][1],data_bathy.variables['spacing'][0])
y = np.arange(data_bathy.variables['y_range'][0],data_bathy.variables['y_range'][1],data_bathy.variables['spacing'][1])
z =data_bathy.variables["z"][:]
z=z.reshape(shape)
z_clean = np.array(z, dtype=float)
z_clean[z_clean > 1e10] = np.nan
z_clean[z_clean < -1e10] = np.nan

#Ombrage
# Remplir les NaN pour le shading
z_filled = np.where(np.isnan(z_clean), np.nanmin(z_clean), z_clean)

dx = data_bathy.variables['spacing'][0]
dy = dx
dy = 111200 * dy
dx = 111200 * dx * np.cos(np.radians(data_bathy.variables['y_range'][0]))

# Colorbar correcte basée sur z (pas sur rgb)
norm = Normalize(vmin=np.nanmin(z_clean), vmax=np.nanmax(z_clean))
z_filled = np.where(np.isnan(z_clean), np.nanmin(z_clean), z_clean)
rgb = ls.shade(z_filled, cmap=cmap, norm=norm, vert_exag=5, blend_mode='hsv', dx=dx, dy=dy)
# Rendre les zones NaN transparentes
mask = np.isnan(z_clean)
rgb[mask] = [1, 1, 1, 0]  # RGBA : blanc totalement transparent

#PLOT
fig, ax = plt.subplots( figsize=(10,10))
ax.imshow(rgb, extent=extent,aspect='auto')
# ax.scatter(catalogue_vaihbav['event_lon'],catalogue_vaihbav['event_lat'],s=1,color= 'darkred',alpha=0.5,label='vaibhav')
# plt.scatter(my_loc_df['lon'],my_loc_df['lat'],s=10,color= 'darkblue',alpha=0.5,label='my_loc')
# plt.scatter(reloc['lon'],reloc['lat'],s=1,color= 'olive',alpha=0.5,label='reloc')
#comparaison
plt.scatter(tmp['lon_x'],tmp['lat_x'],s=10,c='green',alpha=0.8,label='reloc', vmin=0, vmax=10)
plt.scatter(my_loc_df['lon'],my_loc_df['lat'],s=1,color= 'darkblue',alpha=0.1,label='my_loc')
# Colorbar : utiliser un ScalarMappable avec la bonne norm/cmap
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])  # obligatoire mais vide
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('Bathymétrie (m)')

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
zoom = 1
(lat_min, lat_max),(lon_min, lon_max) =(np.float64(-32.22384054054054-zoom), np.float64(-31.123840540540606+zoom)) ,(np.float64(57.75945945945946-zoom), np.float64(58.859459459459394+zoom))
plt.xlim(lon_min, lon_max)
plt.ylim(lat_min,lat_max)
plt.legend()
plt.show()

